1. Imports y configuración

In [7]:
import json
import time
import warnings
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info

warnings.filterwarnings("ignore")

DATASET_DIR   = Path("../../data/processed/dataset/")
CROPS_DIR     = DATASET_DIR / "crops"
MANIFEST_PATH = DATASET_DIR / "manifest.json"
CSV_REVISION  = DATASET_DIR / "revision_manual.csv"
MANIFEST_FINAL = DATASET_DIR / "manifest_labeled.json"

MODEL_ID    = "Qwen/Qwen2.5-VL-3B-Instruct"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 64

print(f"Dispositivo : {DEVICE}")
print(f"PyTorch     : {torch.__version__}")

with open(MANIFEST_PATH, encoding="utf-8") as f:
    manifest = json.load(f)

pendientes = [m for m in manifest if not m.get("label", "").strip()]
print(f"Crops pendientes de etiquetar: {len(pendientes)}")
print(f"Crops ya etiquetados         : {len(manifest) - len(pendientes)}")

Dispositivo : cuda
PyTorch     : 2.7.1+cu118
Crops pendientes de etiquetar: 368
Crops ya etiquetados         : 0


2. Cargar modelo Qwen2.5-VL-3B en 4-bit

In [8]:
print("Cargando modelo en 4-bit (optimizado para 4GB VRAM)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True, 
    bnb_4bit_quant_type="nf4",  
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

print("Modelo cargado correctamente.")
print(f"Memoria GPU usada: {torch.cuda.memory_allocated()/1e9:.2f} GB" 
      if DEVICE == "cuda" else "   Corriendo en CPU.")

Cargando modelo en 4-bit (optimizado para 4GB VRAM)...


ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

3.  Función de transcripción con prompt especializado

In [ ]:
# Prompts por tipo de etiqueta
# Si la etiqueta no está en el diccionario, se usa el prompt genérico
PROMPTS_POR_ETIQUETA = {
    "n_contrato"     : "Transcribe exactly the contract number shown in this handwritten image. Output only the number, nothing else.",
    "fecha"          : "Transcribe exactly the date shown in this handwritten image in Spanish. Output only the date text, nothing else.",
    "contratante"    : "Transcribe exactly the full name shown in this handwritten image. Output only the name in uppercase, nothing else.",
    "fallecido"      : "Transcribe exactly the full name shown in this handwritten image. Output only the name in uppercase, nothing else.",
    "tel_contratante": "Transcribe exactly the phone number shown in this handwritten image. Output only digits, nothing else.",
    "dni_contratante": "Transcribe exactly the ID number shown in this handwritten image. Output only digits, nothing else.",
    "direccion"      : "Transcribe exactly the address shown in this handwritten image in Spanish. Output only the address text, nothing else.",
    "monto"          : "Transcribe exactly the amount shown in this handwritten image. Output only the number with decimals if present, nothing else.",
    "forma_pago"     : "Transcribe exactly the payment method shown in this handwritten image in Spanish. Output only the text, nothing else.",
    "velatorio"      : "Transcribe exactly the word or phrase shown in this handwritten image in Spanish. Output only the text, nothing else.",
    "ataud"          : "Transcribe exactly the text shown in this handwritten image in Spanish. Output only the text, nothing else.",
    "capilla"        : "Transcribe exactly the text shown in this handwritten image in Spanish. Output only the text, nothing else.",
    "carroza"        : "Transcribe exactly the text shown in this handwritten image in Spanish. Output only the text, nothing else.",
    "carroza_flores" : "Transcribe exactly the text shown in this handwritten image in Spanish. Output only the text, nothing else.",
    "cargadores"     : "Transcribe exactly the text shown in this handwritten image in Spanish. Output only the text, nothing else.",
    "vehiculos"      : "Transcribe exactly the text shown in this handwritten image in Spanish. Output only the text, nothing else.",
}

PROMPT_GENERICO = (
    "Transcribe exactly the handwritten text shown in this image in Spanish. "
    "Output only the transcribed text, nothing else."
)


def transcribir_crop(img_path: str, etiqueta: str) -> str:
    """
    Pasa un crop por Qwen2.5-VL y devuelve el texto transcrito.
    """
    prompt = PROMPTS_POR_ETIQUETA.get(etiqueta, PROMPT_GENERICO)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text",  "text": prompt},
            ],
        }
    ]

    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, 
            temperature=None,
            top_p=None,
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return output[0].strip()

4. Transcripción masiva de todos los crops

In [ ]:
if CSV_REVISION.exists():
    df_existente = pd.read_csv(CSV_REVISION, dtype=str).fillna("")
    procesados   = set(df_existente["img_path"].tolist())
    print(f"CSV existente encontrado: {len(procesados)} crops ya procesados.")
    print("Se continuará desde donde se dejó.")
else:
    df_existente = pd.DataFrame()
    procesados   = set()
    print("Iniciando transcripción desde cero.")

nuevas_filas = []
errores      = []

for entrada in tqdm(pendientes, desc="Transcribiendo crops"):
    img_path_rel = entrada["img_path"]

    if img_path_rel in procesados:
        continue

    img_path_abs = str(DATASET_DIR / img_path_rel)

    try:
        texto_qwen = transcribir_crop(img_path_abs, entrada["etiqueta"])
    except Exception as e:
        texto_qwen = ""
        errores.append({"img_path": img_path_rel, "error": str(e)})
        print(f"\n[ERROR] {img_path_rel}: {e}")

    nuevas_filas.append({
        "img_path"        : img_path_rel,
        "etiqueta"        : entrada["etiqueta"],
        "source_img"      : entrada.get("source_img", ""),
        "texto_qwen"      : texto_qwen,
        "texto_corregido" : "",    
        "descartar"       : "",   
    })

df_nuevo = pd.DataFrame(nuevas_filas)
df_final = pd.concat([df_existente, df_nuevo], ignore_index=True)
df_final.to_csv(CSV_REVISION, index=False, encoding="utf-8-sig")
# utf-8-sig: Excel en Windows lo abre correctamente con tildes

print(f"\nCSV guardado en: {CSV_REVISION}")
print(f"   Total filas : {len(df_final)}")
print(f"   Errores     : {len(errores)}")

if errores:
    df_err = pd.DataFrame(errores)
    df_err.to_csv(DATASET_DIR / "errores_transcripcion.csv", index=False)
    print(f"   Log de errores: {DATASET_DIR / 'errores_transcripcion.csv'}")

5. Vista previa del CSV generado

In [ ]:
df_preview = pd.read_csv(CSV_REVISION, dtype=str).fillna("")

print(f"Total de crops a revisar: {len(df_preview)}")
print(f"\nDistribución por etiqueta:")
print(df_preview["etiqueta"].value_counts().to_string())

print(f"\n── Primeras 10 filas ──")
display(df_preview.head(10))

==================== PAUSAR ====================

6. Pausa

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PARA AQUÍ y abre el archivo en Excel:
#
#   revision_manual.csv
#
# Instrucciones:
#   - Columna "texto_qwen"      → lo que transcribió el modelo (NO editar)
#   - Columna "texto_corregido" → escribe aquí solo si Qwen se equivocó
#                                 si está bien, déjala VACÍA
#   - Columna "descartar"       → escribe 1 si el crop es ilegible o inválido
#
# Cuando termines, guarda el CSV y ejecuta la celda 7.
# ═══════════════════════════════════════════════════════════════════════════════

print(f"Abre este archivo en Excel:")
print(f"   {CSV_REVISION.resolve()}")
print()
print("Columnas que debes revisar:")
print("  • texto_qwen      → transcripción automática (solo lectura)")
print("  • texto_corregido → tu corrección (vacío = aceptar texto_qwen)")
print("  • descartar       → escribe 1 si el crop es ilegible")

7.  Importar CSV corregido y generar manifest final

In [ ]:
df_corregido = pd.read_csv(CSV_REVISION, dtype=str).fillna("")

total        = len(df_corregido)
descartados  = (df_corregido["descartar"] == "1").sum()
corregidos   = (df_corregido["texto_corregido"].str.strip() != "").sum()
sin_texto    = (
    (df_corregido["texto_qwen"].str.strip() == "") &
    (df_corregido["texto_corregido"].str.strip() == "") &
    (df_corregido["descartar"] != "1")
).sum()

print(f"Total crops        : {total}")
print(f"Descartados        : {descartados}")
print(f"Corregidos por ti  : {corregidos}")
print(f"Sin texto (alerta) : {sin_texto}")

def resolver_label(row) -> str:
    """
    Prioridad: texto_corregido > texto_qwen
    Si descartar == 1, retorna cadena vacía (se filtrará después).
    """
    if str(row["descartar"]).strip() == "1":
        return ""
    corregido = str(row["texto_corregido"]).strip()
    if corregido:
        return corregido
    return str(row["texto_qwen"]).strip()

df_corregido["label_final"] = df_corregido.apply(resolver_label, axis=1)

label_index = {
    row["img_path"]: row["label_final"]
    for _, row in df_corregido.iterrows()
}

manifest_labeled = []
for entrada in manifest:
    img_path_rel = entrada["img_path"]
    label        = label_index.get(img_path_rel, entrada.get("label", ""))

    if not label:
        continue

    manifest_labeled.append({**entrada, "label": label})

with open(MANIFEST_FINAL, "w", encoding="utf-8") as f:
    json.dump(manifest_labeled, f, ensure_ascii=False, indent=2)

print(f"\nManifest final guardado en: {MANIFEST_FINAL}")
print(f"   Crops con label : {len(manifest_labeled)}")
print(f"   Crops descartados (excluidos): {total - len(manifest_labeled)}")

8. Estadísticas finales del dataset etiquetado

In [ ]:
import matplotlib.pyplot as plt

df_stats = pd.DataFrame(manifest_labeled)

print("═" * 50)
print("DATASET LISTO PARA FINE-TUNING")
print("═" * 50)
print(f"Total muestras          : {len(df_stats)}")
print(f"Etiquetas únicas        : {df_stats['etiqueta'].nunique()}")
print(f"Labels únicos           : {df_stats['label'].nunique()}")
print(f"Largo promedio de texto : {df_stats['label'].str.len().mean():.1f} chars")

print("\nMuestras por etiqueta:")
print(df_stats["etiqueta"].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_stats["etiqueta"].value_counts().plot(
    kind="bar", ax=axes[0], color="#4C72B0", alpha=0.85
)
axes[0].set_title("Muestras por etiqueta")
axes[0].tick_params(axis="x", rotation=45)

df_stats["label"].str.len().hist(bins=20, ax=axes[1], color="#55A868", alpha=0.85)
axes[1].set_title("Distribución de longitud de labels (chars)")
axes[1].set_xlabel("Caracteres")

plt.suptitle("Dataset etiquetado — listo para fine-tuning", fontsize=13)
plt.tight_layout()
plt.savefig(DATASET_DIR / "dataset_stats.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nGráfico guardado en: {DATASET_DIR / 'dataset_stats.png'}")